# Document type classification: fine-tune Gemma 4 E4B

Fine-tune **Gemma 4 E4B** (`google/gemma-4-E4B-it`) on the same 200-row golden dataset used for this task's GEPA prompt-optimization run, evaluated with the **same exact-match metric on the same held-out split**, so results are directly comparable across model families and sizes.

The teacher scores are pending until the owner runs GEPA with a funded key. Do not publish placeholder scores.

**Go criterion:** fine-tuned SLM matches or beats the teacher baseline prompt score.

Runtime: Colab **T4 (free)** works for this model in 4-bit; A100 (Colab Pro) is faster. Runtime -> Change runtime type -> GPU.

In [ ]:
# 0. Hugging Face login FIRST, so the whole run is hands-free after this
# one paste. Use a token with write access (the Gemma license must be
# accepted on this account for the Gemma notebook).
from huggingface_hub import login

login()

In [ ]:
# 1. Install (Colab)
# 2026-07-12 version ledger (each combination tested tonight):
# - unsloth 2026.6.9 + zoo 2026.6.7 + Colab transformers 5.5.0: LOADS
#   gemma-4-E4B on T4 fine. Its KV-sharing guard fires whenever gradient
#   checkpointing is ON (checkpointing's backward recompute IS the "second
#   forward" it forbids -- accumulation=1 did not help), so the model cell
#   disables checkpointing and caps sequences at 2048 to fit the T4.
# - unsloth 2026.7.2: force-upcasts gemma4 to float32 and OOMs a T4 at load.
# - transformers==5.5.2 (the guard's suggested fix): uninstallable -- every
#   unsloth release caps transformers <= 5.5.0 on PyPI.
!pip install -q unsloth==2026.6.9 unsloth-zoo==2026.6.7
import unsloth
import transformers
print(f"unsloth {unsloth.__version__}, transformers {transformers.__version__}")

In [ ]:
# 2. Fetch the same golden dataset (no upload needed) and replicate the GEPA run's exact split
import json
import random
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict

SEED = 42
ROW_COUNT = 200
DATASET = "anirudh1112/corrected-tobacco-dataset-with-ocr"
SPLITS = {"train": 2186, "validation": 272, "test": 279}
PAGE_SIZE = 100
MAX_CONTENT_CHARS = 4000
CLASS_NAMES = ["ADVE", "Email", "Form", "Letter", "Memo", "News", "Note", "Report", "Resume", "Scientific"]
PROMPT_TEMPLATE = """I will provide you with the content and the title of a document.
Your task is to select the most appropriate document type for the document from the list of available document types I will provide.
Only select a document type from the provided list. Respond only with the selected document type name, without any additional information.
If none of the available document types fit the document, respond with an empty string.
The content is likely in English.

The data will be provided using an XML-like format for clarity:

<available_document_types>
{available_document_types}
</available_document_types>

<title>
</title>

<content>
{content}
</content>

Please select the single most appropriate English document type from the list above that best categorizes this document.
Be selective and only choose a document type if it clearly matches the document's nature (e.g., Invoice, Contract, Receipt, Letter, etc.).
"""

def fetch_split(split, total_rows):
    # Authenticated requests get much higher datasets-server rate limits; the
    # login cell above makes get_token() return the pasted token.
    from huggingface_hub import get_token
    token = get_token()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    rows = []
    for offset in range(0, total_rows, PAGE_SIZE):
        params = urllib.parse.urlencode({
            "dataset": DATASET,
            "config": "default",
            "split": split,
            "offset": offset,
            "length": min(PAGE_SIZE, total_rows - offset),
        })
        request = urllib.request.Request(
            f"https://datasets-server.huggingface.co/rows?{params}", headers=headers
        )
        for attempt in range(6):
            try:
                with urllib.request.urlopen(request, timeout=120) as resp:
                    data = json.load(resp)
                break
            except urllib.error.HTTPError as error:
                if error.code in (429, 500, 502, 503) and attempt < 5:
                    wait = 5 * (2 ** attempt)
                    print(f"{split}@{offset}: HTTP {error.code}, retrying in {wait}s")
                    time.sleep(wait)
                else:
                    raise
        rows.extend(item["row"] for item in data["rows"])
        time.sleep(1)  # gentle pacing between pages keeps us under the limit
    return rows

def build_input(text):
    # ponytail: char cap approximates paperless-gpt token truncation for this notebook.
    return PROMPT_TEMPLATE.format(
        available_document_types=", ".join(CLASS_NAMES),
        content=text.strip()[:MAX_CONTENT_CHARS],
    )

source_rows = []
for split_name, total in SPLITS.items():
    source_rows.extend(fetch_split(split_name, total))

by_label = defaultdict(list)
for row in source_rows:
    label = row.get("label", "").strip()
    text = row.get("text", "").strip()
    if label in CLASS_NAMES and text:
        by_label[label].append(row)

rng = random.Random(SEED)
rows = []
for label in CLASS_NAMES:
    rows.extend(rng.sample(by_label[label], ROW_COUNT // len(CLASS_NAMES)))
rng.shuffle(rows)
rows = [{"text": build_input(row["text"]), "expected": row["label"].strip()} for row in rows]

random.Random(SEED).shuffle(rows)
split = int(len(rows) * 0.7)
trainset, devset = rows[:split], rows[split:]
print(f"train {len(trainset)}, held-out {len(devset)}")

In [ ]:
# 3. Same exact-match metric as the GEPA spike (--metric exact)
def exact_match(expected: str, actual: str) -> float:
    return 1.0 if actual.strip().casefold() == expected.strip().casefold() else 0.0


In [ ]:
# 4. Load Gemma 4 E4B in 4-bit + LoRA
# google/gemma-4-E4B-it is natively multimodal (text/image/audio in, text out) but loads and fine-tunes as a plain causal LM for this text-only task.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "google/gemma-4-E4B-it",
    # 2048, not 4096: these prompts tokenize well under 1.5k, and the smaller
    # cap keeps activations inside a T4 with gradient checkpointing OFF below.
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # Checkpointing recomputes the forward during backward -- exactly the
    # "second forward" the Gemma-4 KV-sharing guard forbids on transformers
    # 5.5.0 (it fired even at gradient_accumulation_steps=1). Off is the only
    # installable exit; the shorter sequence cap pays the memory bill.
    use_gradient_checkpointing=False,
)

In [ ]:
# 5. Eval helper + pre-finetune baseline of the raw SLM on the held-out split
import re

def clean_label(s: str) -> str:
    s = re.sub(r"<think>.*?(</think>|$)", "", s, flags=re.DOTALL)
    s = re.sub(r"```.*?```", "", s, flags=re.DOTALL)
    # Strip wrapping quotes BEFORE the emptiness check: a completion that is
    # only quotes ('""') stripped to nothing crashed splitlines()[0] here.
    s = s.strip().strip('"').strip()
    return s.splitlines()[0].strip() if s else ""

def evaluate_slm(model, tokenizer, devset) -> float:
    FastLanguageModel.for_inference(model)
    scores = []
    for row in devset:
        messages = [{"role": "user", "content": row["text"]}]
        try:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
        # text= keyword, not a positional arg: multimodal processors may treat a positional arg as an image.
        inputs = tokenizer(text=prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        scores.append(exact_match(row["expected"], clean_label(completion)))
    return 100 * sum(scores) / len(scores)

raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Gemma 4 E4B raw (no fine-tune): {raw_slm_score:.2f}")


In [ ]:
# 6. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_text(row):
    messages = [
        {"role": "user", "content": row["text"]},
        {"role": "assistant", "content": row["expected"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        # accumulation=1 dodges the Gemma-4 E-series KV-sharing guard on
        # transformers 5.5.0 (it forbids summed microbatch losses with
        # gradient checkpointing; single forward per backward is fine).
        # Effective batch is 2, not 8; drop learning_rate to 1e-4 if the
        # loss curve looks unstable.
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 7. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)

gpt54_baseline = 78.33  # gpt-5.4-mini, verbatim shipped paperless-gpt prompt (README, 2026-07-10)
gpt54_optimized = 81.67  # same run, GEPA-optimized

print("=" * 52)
print(f"gpt-5.4-mini baseline prompt : {gpt54_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54_optimized:.2f}")
print(f"Gemma 4 E4B raw               : {raw_slm_score:.2f}")
print(f"Gemma 4 E4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= teacher baseline prompt score")


In [ ]:
# 8. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive)
model.save_pretrained("apprentice-gemma4-e4b-lora-document-types")
tokenizer.save_pretrained("apprentice-gemma4-e4b-lora-document-types")
print("saved -> apprentice-gemma4-e4b-lora-document-types/")


In [ ]:
# 8b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 7 before pushing. Never push a card
# with a number that was not actually printed by this run.
HF_USERNAME = "singhabhishekkk"
REPO_ID = f"{HF_USERNAME}/apprentice-gemma4-e4b-lora-document-types"

model_card = "---\ndatasets: [anirudh1112/corrected-tobacco-dataset-with-ocr]\nbase_model: google/gemma-4-E4B-it\nlibrary_name: peft\nlicense: apache-2.0\ntags: [lora, unsloth, gemma, document-type-classification, apprentice, paperless-gpt]\n---\n\n# Apprentice Gemma 4 E4B LoRA (document type classification)\n\nLoRA adapter fine-tuned on 140 golden examples from a corrected Tobacco3482 OCR-text dataset to classify one document type from: ADVE, Email, Form, Letter, Memo, News, Note, Report, Resume, Scientific.\n\nThe input prompt is the verbatim shipped document-type prompt from icereed/paperless-gpt, filled with English, the allowed type list, an empty title, and OCR text capped at about 4,000 characters.\n\n## Results (60 held-out rows, exact match)\n\nFill in from this task's README after publishing this run's printed numbers.\n\n## Training\n\nLoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit, Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows, identical split across every model this task is fine-tuned on.\n\n## Usage\n\nLoad with PEFT on top of `google/gemma-4-E4B-it`, or serve locally with an adapter-capable runtime. Caveat: evaluated on 60 rows for one field only. Re-validate on your paperless-ngx document types before production use.\n"

with open("apprentice-gemma4-e4b-lora-document-types/README.md", "w") as f:
    f.write(model_card)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, private=False, exist_ok=True)
api.upload_folder(folder_path=REPO_ID.split('/')[1], repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")


## After this run

1. Publish the two SLM numbers exactly as printed. This repo never carries a number that was not measured.
2. Fill in the model card's results table with the real GEPA numbers from this task's README before pushing.
3. If APIs error: Unsloth moves fast. Cross-check against the current official Unsloth notebook for this model.